### Import libraries

In [12]:
import os, json, glob, gc, time
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import collections.abc

from darts import TimeSeries
from darts.dataprocessing.transformers import StaticCovariatesTransformer
from darts.models import TFTModel
from pytorch_lightning.callbacks import EarlyStopping


### Config

In [13]:
local_test_dir = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Data_range_from_Apr_23\Chunking\local_test_data"   # use absolute path if notebooks differ in cwd
local_train_dir  = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Data_range_from_Apr_23\Chunking\local_train_data"
# group_keys_path = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Data_range_from_Apr_23\Chunking\saved_group_keys\group_keys.json"

CACHE_DIR       = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Data_range_from_Apr_23\Chunking\series_cache"          # per-series .npz files live here

time_col   = 'CAL_DATE'
group_col  = 'PARENT_DEALER_CODE_MODEL_FAMILY'
target_col = 'NET_SALES'
FREQ       = 'D'

TRAIN_END = pd.Timestamp("2025-12-31")
VAL_START = pd.Timestamp("2026-01-01")
VAL_END   = pd.Timestamp("2026-06-30")

INPUT_CHUNK_LENGTH  = 365
OUTPUT_CHUNK_LENGTH = 184
TEST_HORIZON        = 184

static_covariates = [
    'PARENT_DEALER_CODE', 'MODEL_FAMILY', 'MODEL_NAME', 'BRAKE_TYPE',
    'IGNITION_TYPE', 'WHEEL_TYPE', 'COLOUR', 'DEALER_CITY',
    'X_CITY_CATEGORY', 'ZONAL_OFFICE_NAME'
]

future_covariates = [
    'NEW_YEAR','LOHRI','MAKAR_SANKRANTI','REPUBLIC_DAY','VASANT_PANCHAMI',
    'MAHA_SHIVRATRI','EID_UL_FITR','HOLIKA_DAHAN','HOLI','HANUMAN_JAYANTI',
    'AKSHAYA_TRITYA','BUDDHA_PURNIMA','GANGA_DUSSEHRA','JAGANNATH_RATHYATRA',
    'GURU_PURNIMA','NAG_PANCHAMI','RAKSHA_BANDHAN','HARTALIK_TEEJ',
    'GANESH_CHATURTHI','JANMASHTAMI','VISHWAKARMA_PUJA','KARWA_CHAUTH',
    'ONAM','MARRIAGE_DAY',
    'N-16','N-15','N-14','N-13','N-12','N-11','N-10','N-9','N-8','N-7',
    'N-6','N-5','N-4','N-3','N-2','N-1','N','N+1','N+2','N+3','N+4',
    'N+5','N+6','N+7','N+8','N+9','N+10',
    'D-3','D-2','D-1','D','D+1','D+2','D+3','D+4','D+5','D+6',
    'C','C+1','C+2','C+3','C+4','C+5','C+6'
]

penalty_cols = [
    'N-16','N-15','N-14','N-13','N-12','N-11','N-10','N-9','N-8','N-7',
    'N-6','N-5','N-4','N-3','N-2','N-1','N','N+1','N+2','N+3','N+4',
    'N+5','N+6','N+7','N+8','N+9','N+10',
    'D-3','D-2','D-1','D','D+1','D+2','D+3','D+4','D+5','D+6',
    'C','C+1','C+2','C+3','C+4','C+5','C+6'
]

val_window_days = (VAL_END - VAL_START).days + 1                       # 181
warmup_days     = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH - val_window_days  # 368
warmup_start    = VAL_START - pd.Timedelta(days=warmup_days)
MIN_LEN         = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH             # 549


### Custom loss function

In [14]:
class HuberMaeFeatureLoss(nn.HuberLoss):
    """
    Total loss = Huber(y_hat, y) + flag * MAE(y_hat, y)
    computed only on component 0. Component 1 of the target is the flag.
    On festive days (flag=1) the absolute error is counted twice in effect:
    once inside Huber, once as the added MAE penalty.
    """
    def __init__(self, delta=1.0, reduction='mean'):
        super().__init__(reduction='none', delta=delta)
        self.user_reduction = reduction

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # Shapes: (batch, timesteps, n_components)
        # Component 0 = NET_SALES, component 1 = FESTIVE_FLAG
        y_hat = input[..., 0]
        y     = target[..., 0]
        flag  = target[..., 1]

        huber_base = super().forward(y_hat, y)

        mae = torch.abs(y - y_hat)

        total_loss = huber_base + flag * mae

        if self.user_reduction == 'mean':
            return total_loss.mean()
        elif self.user_reduction == 'sum':
            return total_loss.sum()
        return total_loss


In [15]:

print("="*60)
print("SECTION 2: BUILDING PER-SERIES CACHE")
print("="*60)

os.makedirs(CACHE_DIR, exist_ok=True)
manifest_path = os.path.join(CACHE_DIR, "manifest.json")

def safe_name(key):
    return str(key).replace("<>", "_").replace("/", "_").replace("\\", "_")

if os.path.exists(manifest_path):
    print("Cache already exists — loading manifest.")
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
else:
    needed_cols = [time_col, group_col, target_col] + static_covariates + penalty_cols
    chunk_files = sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet")))
    print(f"Scanning {len(chunk_files)} chunk files...")

    series_keys  = []
    has_val      = []
    scaler_stats = {}
    static_rows  = []

    for ci, chunk_path in enumerate(chunk_files):
        df = pd.read_parquet(chunk_path, columns=needed_cols)
        df[time_col] = pd.to_datetime(df[time_col])

        for key, g in df.groupby(group_col, sort=False):
            g = g.sort_values(time_col).reset_index(drop=True)
            flag = (g[penalty_cols] != 0).any(axis=1).to_numpy(dtype=np.float32)

            t = g[time_col]
            tr = (t <= TRAIN_END).to_numpy()
            va = ((t >= warmup_start) & (t <= VAL_END)).to_numpy()

            if tr.sum() < MIN_LEN:
                continue   # too short to yield even one training sample

            sales = g[target_col].to_numpy(dtype=np.float32)
            tr_sales, tr_flag = sales[tr], flag[tr]

            # scaling params from TRAIN window only (no leakage from val)
            lo = float(tr_sales.min())
            hi = float(tr_sales.max())
            if hi - lo < 1e-8:
                hi = lo + 1.0            # constant series guard

            keep_val = va.sum() >= MIN_LEN
            payload = {
                "train_sales": tr_sales,
                "train_flag":  tr_flag,
                "train_start": np.array(str(t[tr].iloc[0].date())),
            }
            if keep_val:
                payload["val_sales"] = sales[va]
                payload["val_flag"]  = flag[va]
                payload["val_start"] = np.array(str(t[va].iloc[0].date()))

            np.savez(os.path.join(CACHE_DIR, f"{safe_name(key)}.npz"), **payload)

            series_keys.append(str(key))
            has_val.append(bool(keep_val))
            scaler_stats[str(key)] = [lo, hi]
            static_rows.append(g[static_covariates].iloc[0].to_dict())

        del df
        gc.collect()
        print(f"  chunk {ci+1}/{len(chunk_files)} done — series so far: {len(series_keys)}")

    pd.DataFrame(static_rows).to_parquet(
        os.path.join(CACHE_DIR, "static_covariates.parquet"), index=False
    )
    manifest = {"series_keys": series_keys, "has_val": has_val,
                "scaler_stats": scaler_stats}
    with open(manifest_path, "w") as f:
        json.dump(manifest, f)
    del static_rows
    gc.collect()

series_keys  = manifest["series_keys"]
has_val      = manifest["has_val"]
scaler_stats = manifest["scaler_stats"]
print(f"\nSeries cached          : {len(series_keys)}")
print(f"Series with val strip  : {sum(has_val)}")


SECTION 2: BUILDING PER-SERIES CACHE
Cache already exists — loading manifest.

Series cached          : 117246
Series with val strip  : 117246


In [16]:
print("\n" + "="*60)
print("SECTION 3: STATIC COVARIATES")
print("="*60)

from sklearn.preprocessing import OrdinalEncoder

static_df_all = pd.read_parquet(os.path.join(CACHE_DIR, "static_covariates.parquet"))

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoded_arr = encoder.fit_transform(
    static_df_all[static_covariates].astype(str)
).astype(np.float32)

encoded_df = pd.DataFrame(encoded_arr, columns=static_covariates)
STATIC_ENCODED = [
    encoded_df.iloc[[i]].reset_index(drop=True)
    for i in range(len(encoded_df))
]

print(f"Encoded static covariates for {len(STATIC_ENCODED)} series.")
for c in static_covariates:
    print(f"  {c}: {static_df_all[c].nunique()} categories")



SECTION 3: STATIC COVARIATES
Encoded static covariates for 117246 series.
  PARENT_DEALER_CODE: 1176 categories
  MODEL_FAMILY: 14 categories
  MODEL_NAME: 14 categories
  BRAKE_TYPE: 2 categories
  IGNITION_TYPE: 2 categories
  WHEEL_TYPE: 4 categories
  COLOUR: 31 categories
  DEALER_CITY: 841 categories
  X_CITY_CATEGORY: 4 categories
  ZONAL_OFFICE_NAME: 6 categories


In [17]:
print("\n" + "="*60)
print("SECTION 4: SHARED FUTURE COVARIATES")
print("="*60)

cov_cols = [time_col] + future_covariates
train_chunk0 = sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet")))[0]
test_chunk0  = sorted(glob.glob(os.path.join(local_test_dir,  "chunk_*.parquet")))[0]

cal = pd.concat([
    pd.read_parquet(train_chunk0, columns=cov_cols),
    pd.read_parquet(test_chunk0,  columns=cov_cols),
])
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.drop_duplicates(subset=time_col).sort_values(time_col).reset_index(drop=True)

SHARED_COV = TimeSeries.from_dataframe(
    cal, time_col=time_col, value_cols=future_covariates,
    freq=FREQ, fill_missing_dates=False
).astype(np.float32)

print(f"Covariate calendar: {cal[time_col].min().date()} → {cal[time_col].max().date()} "
      f"({len(cal)} days, {len(future_covariates)} cols)")
del cal
gc.collect()



SECTION 4: SHARED FUTURE COVARIATES
Covariate calendar: 2023-04-01 → 2026-12-31 (1371 days, 68 cols)


0

In [18]:

class DiskLazyTargetSequence(collections.abc.Sequence):
    """
    Target series backed by per-series .npz files.

    Returns a 2-component TimeSeries: [scaled NET_SALES, raw FESTIVE_FLAG].
    Scaling is applied at read time using train-window min/max.

    With cache_in_ram=True, arrays are held after first read. The cache is
    populated after worker fork, so it is never pickled — worker spawn stays
    cheap while per-epoch disk I/O disappears after epoch 1.
    """

    def __init__(self, cache_dir, series_keys, scaler_stats, static_encoded,
                 split="train", freq='D', cache_in_ram=True):
        self.cache_dir      = cache_dir
        self.series_keys    = series_keys
        self.scaler_stats   = scaler_stats
        self.static_encoded = static_encoded
        self.split          = split
        self.freq           = freq
        self.cache_in_ram   = cache_in_ram
        self._ram           = {} if cache_in_ram else None

    def __len__(self):
        return len(self.series_keys)

    # keep the RAM cache out of anything pickled to workers
    def __getstate__(self):
        state = self.__dict__.copy()
        state["_ram"] = {} if self.cache_in_ram else None
        return state

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return [self[i] for i in range(*idx.indices(len(self)))]
        if idx < 0:
            idx += len(self)
        if not 0 <= idx < len(self):
            raise IndexError(idx)

        if self._ram is not None and idx in self._ram:
            sales, flag, start = self._ram[idx]
        else:
            key  = self.series_keys[idx]
            path = os.path.join(self.cache_dir, f"{safe_name(key)}.npz")
            with np.load(path, allow_pickle=False) as z:
                sales = z[f"{self.split}_sales"]
                flag  = z[f"{self.split}_flag"]
                start = str(z[f"{self.split}_start"])
            if self._ram is not None:
                self._ram[idx] = (sales, flag, start)

        lo, hi = self.scaler_stats[self.series_keys[idx]]
        scaled = ((sales - lo) / (hi - lo)).astype(np.float32)

        values = np.stack([scaled, flag], axis=1)
        times  = pd.date_range(start=start, periods=len(values), freq=self.freq)

        return TimeSeries.from_times_and_values(
            times, values,
            columns=[target_col, "FESTIVE_FLAG"],
            static_covariates=self.static_encoded[idx],
        )
    
class SharedCovSequence(collections.abc.Sequence):
    """Returns the same in-RAM covariate TimeSeries for every index."""

    def __init__(self, shared_series, n):
        self.shared = shared_series
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return [self.shared for _ in range(*idx.indices(self.n))]
        if idx < 0:
            idx += self.n
        if not 0 <= idx < self.n:
            raise IndexError(idx)
        return self.shared


# --- train sequences (all series) ---
train_seq = DiskLazyTargetSequence(
    CACHE_DIR, series_keys, scaler_stats, STATIC_ENCODED, split="train", freq=FREQ
)
train_cov_seq = SharedCovSequence(SHARED_COV, len(series_keys))

# --- val sequences (only series that have a val strip) ---
val_keys    = [k for k, h in zip(series_keys, has_val) if h]
val_statics = [s for s, h in zip(STATIC_ENCODED, has_val) if h]
val_seq = DiskLazyTargetSequence(
    CACHE_DIR, val_keys, scaler_stats, val_statics, split="val", freq=FREQ
)
val_cov_seq = SharedCovSequence(SHARED_COV, len(val_keys))

print("\n" + "="*60)
print("SECTION 5: SANITY CHECK")
print("="*60)
_t0 = time.time()
s0 = train_seq[0]
print(f"train_seq[0]: {s0.start_time().date()} → {s0.end_time().date()} | "
      f"len={len(s0)} | comps={s0.components.tolist()}")
print(f"single lazy read took {(time.time()-_t0)*1000:.1f} ms")
print(f"train series: {len(train_seq)} | val series: {len(val_seq)}")

# Read-speed probe: this number decides whether disk-lazy is viable at all
_t0 = time.time()
for i in np.random.randint(0, len(train_seq), 200):
    _ = train_seq[int(i)]
_per_read = (time.time() - _t0) / 200
print(f"\nAvg random read: {_per_read*1000:.2f} ms")
print(f"→ est. time for 1 epoch at 50 samples/series: "
      f"{_per_read * len(train_seq) * 50 / 3600:.1f} h of pure disk I/O")


SECTION 5: SANITY CHECK
train_seq[0]: 2023-04-01 → 2025-12-31 | len=1006 | comps=['NET_SALES', 'FESTIVE_FLAG']
single lazy read took 12.2 ms
train series: 117246 | val series: 117246

Avg random read: 1.32 ms
→ est. time for 1 epoch at 50 samples/series: 2.2 h of pure disk I/O


In [19]:
class HuberMaeFeatureLoss(nn.HuberLoss):
    """Huber(y_hat, y) + flag * |y - y_hat|, on component 0 only.
    Component 1 of the target carries the festive flag."""

    def __init__(self, delta=1.0, reduction='mean'):
        super().__init__(reduction='none', delta=delta)
        self.user_reduction = reduction

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        y_hat = input[..., 0]
        y     = target[..., 0]
        flag  = target[..., 1]

        total = super().forward(y_hat, y) + flag * torch.abs(y - y_hat)

        if self.user_reduction == 'mean':
            return total.mean()
        if self.user_reduction == 'sum':
            return total.sum()
        return total


In [20]:
print("\n" + "="*60)
print("SECTION 7: MODEL")
print("="*60)

torch.set_float32_matmul_precision('high')

now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")
MODEL_NAME = f"daily_tft_festive_lazy_{now}"
print("Model name:", MODEL_NAME)

early_stopping = EarlyStopping(
    monitor="val_loss", patience=10, min_delta=1e-4, mode="min"
)

model = TFTModel(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,

    hidden_size=32,
    lstm_layers=4,
    num_attention_heads=16,
    dropout=0.05,

    batch_size=256,
    n_epochs=100,

    likelihood=None,
    loss_fn=HuberMaeFeatureLoss(delta=1.0, reduction='mean'),

    random_state=42,
    add_relative_index=True,

    save_checkpoints=True,
    force_reset=True,
    model_name=MODEL_NAME,
    skip_interpolation=True,

    pl_trainer_kwargs={
        "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
        "devices": 1,
        "callbacks": [early_stopping],
        "gradient_clip_val": 0.1,
        "precision": "bf16-mixed",
    },
)



SECTION 7: MODEL
Model name: daily_tft_festive_lazy_2026-07-21_12_48_20


### RAM logging

In [21]:
import psutil, threading, time, os

_proc = psutil.Process(os.getpid())
_stop = threading.Event()

def _log_ram(interval=300):
    peak = 0
    while not _stop.is_set():
        rss  = _proc.memory_info().rss / 1e9
        kids = sum(c.memory_info().rss for c in _proc.children(recursive=True)) / 1e9
        vm   = psutil.virtual_memory()
        peak = max(peak, rss + kids)
        print(f"[{time.strftime('%H:%M:%S')}] main: {rss:.1f} GB | "
              f"workers: {kids:.1f} GB | total: {rss+kids:.1f} GB (peak {peak:.1f}) | "
              f"system: {vm.used/1e9:.1f}/{vm.total/1e9:.1f} GB", flush=True)
        _stop.wait(interval)

_stop.clear()
threading.Thread(target=_log_ram, daemon=True).start()
print("RAM logger started.")


[12:48:20] main: 8.5 GB | workers: 0.0 GB | total: 8.5 GB (peak 8.5) | system: 29.2/67.9 GB
RAM logger started.


In [22]:
print("\n" + "="*60)
print("SECTION 8: TRAINING")
print("="*60)

try:
    model.fit(
        series=train_seq,
        future_covariates=train_cov_seq,

        val_series=val_seq,
        val_future_covariates=val_cov_seq,

        max_samples_per_ts=50,

        dataloader_kwargs={
            "num_workers": 0,
            "pin_memory": True,
        },
        verbose=True,
    )
finally:
    _stop.set()
    print("RAM logger stopped.")


SECTION 8: TRAINING


Detected user-defined float16-like precision. For mixed precision training, recommended options are 'bf16-mixed' and '16-mixed'.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | criterion                         | HuberMaeFeatureLoss              | 0      | train
1  | train_criterion                   | HuberMaeFeatureLoss              | 0      | train
2  | val_criterion                     | HuberMaeFeatureLoss              | 0      | train
3  | train_metrics                     | MetricCollection                 | 0      | train
4  | val_metrics                       | MetricCollection                 | 0      | train
5  | input_embeddings 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

[12:53:20] main: 9.8 GB | workers: 0.0 GB | total: 9.8 GB (peak 9.8) | system: 30.6/67.9 GB


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

[16:58:34] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.4/67.9 GB
[17:03:34] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.4/67.9 GB
[17:08:34] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.4/67.9 GB
[17:13:34] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.5/67.9 GB
[17:18:35] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.5/67.9 GB
[17:23:35] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.5/67.9 GB
[17:28:35] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.5/67.9 GB
[17:33:35] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.3/67.9 GB
[17:38:35] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.3/67.9 GB
[17:43:35] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) | system: 29.3/67.9 GB
[17:48:35] main: 3.3 GB | workers: 0.0 GB | total: 3.3 GB (peak 11.7) 


Detected KeyboardInterrupt, attempting graceful shutdown ...


RAM logger stopped.


NameError: name 'exit' is not defined

### Prediction

In [23]:
SHARED_COV.to_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
print("Saved:", SHARED_COV.start_time().date(), "→", SHARED_COV.end_time().date())

Saved: 2023-04-01 → 2026-12-31


In [24]:
DATA_ROOT = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Data_range_from_Apr_23\Chunking"
CACHE_DIR  = os.path.join(DATA_ROOT, "series_cache")
OUT_DIR    = os.path.join(DATA_ROOT, "predictions_2026")

In [25]:
MODEL_NAME

'daily_tft_festive_lazy_2026-07-21_12_48_20'